# Task 2 Lineage-Holdout Significance Evaluation

Tests whether the lineage-holdout improvement in Precision@10/Recall@10 for WHO-catalogued
variant recovery (`FINAL_controlled_shap_comparison.csv`) is statistically significant, per
architecture, and applies a Holm-Bonferroni correction across the 4 architectures within
each metric (precision, recall).

In [1]:
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

INPUT_FILE = "data/latest/lineage_ood_all_train/FINAL_controlled_shap_comparison.csv"
OUT_FILE = "data/latest/lineage_ood_all_train/task2_lineage_significance_holm.csv"


In [2]:
df = pd.read_csv(INPUT_FILE)

# Paired Wilcoxon signed-rank test (two-sided) per architecture, pairing each drug's
# lineage-holdout-controlled Precision@10 (and Recall@10) mean against that drug's random-baseline
# value, across all 8 drugs.
rows = []
for arch in sorted(df["architecture"].unique()):
    sub = df[df["architecture"] == arch].sort_values("drug")
    _, p_precision10 = wilcoxon(sub["lineage_controlled_p10_mean"], sub["random_p10"], alternative="two-sided")
    _, p_recall10 = wilcoxon(sub["lineage_controlled_r10_mean"], sub["random_r10"], alternative="two-sided")
    rows.append({
        "architecture": arch,
        "mean_delta_p10": sub["delta_p10"].mean(),
        "mean_delta_r10": sub["delta_r10"].mean(),
        "p_precision10": p_precision10,
        "p_recall10": p_recall10,
        "n_drugs": len(sub),
    })

result = pd.DataFrame(rows)

# Holm-Bonferroni correction, applied separately within each metric's family of 4 architecture-level tests.
result["p_precision10_holm"] = multipletests(result["p_precision10"], method="holm")[1]
result["p_recall10_holm"] = multipletests(result["p_recall10"], method="holm")[1]

result.to_csv(OUT_FILE, index=False)
print(f"[OK] Wrote {OUT_FILE} with {len(result)} rows.")
result

[OK] Wrote data/latest/lineage_ood_all_train/task2_lineage_significance_holm.csv with 4 rows.


/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


,architecture,mean_delta_p10,mean_delta_r10,p_precision10,p_recall10,n_drugs,p_precision10_holm,p_recall10_holm
0,cnn,0.015625,0.03275,0.461838,0.382812,8,0.923676,0.746592
1,esm_full320,0.096875,0.16550,0.046399,0.027708,8,0.185598,0.110831
2,regression,0.028125,0.02750,0.463071,0.345448,8,0.923676,0.746592
3,transformer,0.068750,0.03150,0.074150,0.248864,8,0.222450,0.746592
